In [1]:
pip install duckdb

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np 
import pandas as pd

In [4]:
import requests
import json

In [5]:
url = "https://api.worldbank.org/v2/country?format=json&per_page=300"
response = requests.get(url)
countries = response.json()[1]

In [6]:
# check the HTTP status. 200 means success.
print("Status code:", response.status_code)

Status code: 200


In [7]:
# turn the raw response into Python objects
real = [c for c in countries if c["region"]["value"] != "Aggregates"]
agg  = [c for c in countries if c["region"]["value"] == "Aggregates"]

In [8]:
print("Real countries:", len(real))
print("Aggregates:", len(agg))


Real countries: 217
Aggregates: 79


In [9]:
with open("countries_real.json", "w") as f:
    json.dump(real, f)

In [10]:
print("Saved", len(real), "countries")

Saved 217 countries


In [11]:
import os
print(os.path.exists("countries_real.json"))
print(os.path.getsize("countries_real.json"), "bytes")

True
95134 bytes


In [12]:
#  Fetching indicators
base = "https://api.worldbank.org/v2/indicator?format=json&per_page=5000"

all_inds = []
r = requests.get(base + "&page=1")
meta = r.json()[0]
all_inds.extend(r.json()[1])
pages = meta["pages"]
print("Pages:", pages, "| page 1:", len(all_inds))
for p in range(2, pages + 1):
    chunk = requests.get(base + f"&page={p}").json()[1]
    all_inds.extend(chunk)
    print(f"page {p}: {len(chunk)}")

print("Total:", len(all_inds))

Pages: 6 | page 1: 5000
page 2: 5000
page 3: 5000
page 4: 5000
page 5: 5000
page 6: 4544
Total: 29544


In [13]:
import csv

with open("indicators_catalog.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["indicator_id", "indicator_name", "definition", "source"])
    for ind in all_inds:
        writer.writerow([
            ind.get("id", ""),
            ind.get("name", ""),
            ind.get("sourceNote", ""),
            ind.get("source", {}).get("value", "")
        ])

print("Wrote indicators_catalog.csv with", len(all_inds), "rows")

Wrote indicators_catalog.csv with 29544 rows


In [14]:
import pandas as pd
df_ind = pd.read_csv("indicators_catalog.csv")
print(df_ind.shape)
print(df_ind["source"].value_counts().head(10))

(29544, 4)
source
Education Statistics                                                   8317
The Atlas of Social Protection: Indicators of Resilience and Equity    3561
Global Findex database                                                 2947
Quarterly External Debt Statistics SDDS                                1800
Disability Data Hub (DDH)                                              1577
World Development Indicators                                           1499
WDI Database Archives                                                  1017
Gender Statistics                                                       948
Africa Development Indicators                                           837
Global Jobs Indicators Database (JOIN)                                  746
Name: count, dtype: int64


In [15]:
wdi = df_ind[df_ind["source"] == "World Development Indicators"].copy()
print(wdi.shape)
print(wdi.head())

(1499, 4)
           indicator_id                                     indicator_name  \
1337  AG.CON.FERT.PT.ZS  Fertilizer consumption (% of fertilizer produc...   
1338     AG.CON.FERT.ZS  Fertilizer consumption (kilograms per hectare ...   
1358     AG.LND.AGRI.K2                         Agricultural land (sq. km)   
1359     AG.LND.AGRI.ZS                 Agricultural land (% of land area)   
1360     AG.LND.ARBL.HA                             Arable land (hectares)   

                                             definition  \
1337  Fertilizer consumption measures the quantity o...   
1338  Fertilizer consumption measures the quantity o...   
1358  Agricultural land refers to the land area that...   
1359  Agricultural land refers to the share of land ...   
1360  Arable land (in hectares) includes land define...   

                            source  
1337  World Development Indicators  
1338  World Development Indicators  
1358  World Development Indicators  
1359  World Develo

In [16]:
keywords = [
    "Individuals using the Internet",
    "Fixed broadband subscriptions",
    "Mobile cellular subscriptions",
    "Secure Internet servers",
    "ICT service exports",
    "Investment in ICT with private participation",
    "Account ownership at a financial institution",
    "GDP per person employed",
    "Labor force participation rate, female",
    "Ratio of female to male labor force participation",
    "GDP per capita, PPP",
    "School enrollment, tertiary",
    "Trade (% of GDP)",
    "Urban population",
    "Access to electricity",
    "Government effectiveness",
    "Regulatory quality",
]

for kw in keywords:
    hits = wdi[wdi["indicator_name"].str.contains(kw, case=False, na=False)]
    print(f"--- {kw} ({len(hits)} hits)")
    for _, r in hits.iterrows():
        print("   ", r["indicator_id"], "|", r["indicator_name"])

--- Individuals using the Internet (3 hits)
    IT.NET.USER.FE.ZS | Individuals using the Internet, female (% of female population)
    IT.NET.USER.MA.ZS | Individuals using the Internet, male (% of male population)
    IT.NET.USER.ZS | Individuals using the Internet (% of population)
--- Fixed broadband subscriptions (2 hits)
    IT.NET.BBND | Fixed broadband subscriptions
    IT.NET.BBND.P2 | Fixed broadband subscriptions (per 100 people)
--- Mobile cellular subscriptions (2 hits)
    IT.CEL.SETS | Mobile cellular subscriptions
    IT.CEL.SETS.P2 | Mobile cellular subscriptions (per 100 people)
--- Secure Internet servers (2 hits)
    IT.NET.SECR | Secure Internet servers
    IT.NET.SECR.P6 | Secure Internet servers (per 1 million people)
--- ICT service exports (2 hits)
    BX.GSR.CCIS.CD | ICT service exports (BoP, current US$)
    BX.GSR.CCIS.ZS | ICT service exports (% of service exports, BoP)
--- Investment in ICT with private participation (1 hits)
    IE.PPI.ICTI.CD | Investme

/var/folders/6v/s9_yhnl534n43tphxkf70pxh0000gn/T/ipykernel_41229/3880633887.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = wdi[wdi["indicator_name"].str.contains(kw, case=False, na=False)]


In [17]:
import re

wdi[wdi["indicator_name"].str.contains(re.escape("Trade (% of GDP)"), case=False, na=False)][["indicator_id", "indicator_name"]]

,indicator_id,indicator_name
15837,NE.TRD.GNFS.ZS,Trade (% of GDP)
27089,TG.VAL.TOTL.GD.ZS,Merchandise trade (% of GDP)


In [18]:
wdi[wdi["indicator_id"].isin(["GOV_WGI_GE_EST", "FX.OWN.TOTL.ZS", "IT.NET.USER.ZS", "SL.TLF.CACT.FE.ZS"])][["indicator_id", "indicator_name", "source"]]

,indicator_id,indicator_name,source
11117,FX.OWN.TOTL.ZS,Account ownership at a financial institution o...,World Development Indicators
11666,GOV_WGI_GE_EST,Government Effectiveness - Governance estimate...,World Development Indicators
13546,IT.NET.USER.ZS,Individuals using the Internet (% of population),World Development Indicators
26387,SL.TLF.CACT.FE.ZS,"Labor force participation rate, female (% of f...",World Development Indicators


In [19]:
shortlist = [
    "IT.NET.USER.ZS", "IT.NET.USER.FE.ZS", "IT.NET.USER.MA.ZS",
    "IT.NET.BBND.P2", "IT.CEL.SETS.P2", "IT.NET.SECR.P6",
    "BX.GSR.CCIS.ZS", "IE.PPI.ICTI.CD",
    "FX.OWN.TOTL.ZS", "FX.OWN.TOTL.FE.ZS",
    "SL.GDP.PCAP.EM.KD", "SL.TLF.CACT.FE.ZS", "SL.TLF.CACT.FM.ZS",
    "NY.GDP.PCAP.PP.KD", "SE.TER.ENRR", "NE.TRD.GNFS.ZS",
    "SP.URB.TOTL.IN.ZS", "EG.ELC.ACCS.ZS",
    "GOV_WGI_GE_EST", "GOV_WGI_RQ_EST",
]

with open("shortlist_codes.txt", "w") as f:
    f.write("\n".join(shortlist))

print(len(shortlist), "codes saved")

20 codes saved


In [20]:
import time

with open("countries_real.json") as f:
    countries = json.load(f)
country_codes = [c["id"] for c in countries]

shortlist = [
    "IT.NET.USER.ZS", "IT.NET.USER.FE.ZS", "IT.NET.USER.MA.ZS",
    "IT.NET.BBND.P2", "IT.CEL.SETS.P2", "IT.NET.SECR.P6",
    "BX.GSR.CCIS.ZS", "IE.PPI.ICTI.CD",
    "FX.OWN.TOTL.ZS", "FX.OWN.TOTL.FE.ZS",
    "SL.GDP.PCAP.EM.KD", "SL.TLF.CACT.FE.ZS", "SL.TLF.CACT.FM.ZS",
    "NY.GDP.PCAP.PP.KD", "SE.TER.ENRR", "NE.TRD.GNFS.ZS",
    "SP.URB.TOTL.IN.ZS", "EG.ELC.ACCS.ZS",
    "GOV_WGI_GE_EST", "GOV_WGI_RQ_EST",
]

all_rows = []

for code in shortlist:
    url = f"https://api.worldbank.org/v2/country/all/indicator/{code}?format=json&per_page=20000&date=1990:2025"
    r = requests.get(url)
    if r.status_code != 200:
        print(code, "failed", r.status_code)
        continue
    data = r.json()
    if len(data) < 2 or data[1] is None:
        print(code, "no data")
        continue
    rows = data[1]
    print(f"{code}: {len(rows)} of {data[0]['total']}")
    for row in rows:
        if row["countryiso3code"] in country_codes and row["value"] is not None:
            all_rows.append({
                "country": row["countryiso3code"],
                "year": int(row["date"]),
                "indicator": row["indicator"]["id"],
                "value": row["value"]
            })
    time.sleep(0.3)

df_long = pd.DataFrame(all_rows)
df_long.to_csv("panel_long.csv", index=False)
print("Total observations:", len(df_long))

IT.NET.USER.ZS: 9576 of 9576
IT.NET.USER.FE.ZS: 9576 of 9576
IT.NET.USER.MA.ZS: 9576 of 9576
IT.NET.BBND.P2: 9576 of 9576
IT.CEL.SETS.P2: 9576 of 9576
IT.NET.SECR.P6: 9576 of 9576
BX.GSR.CCIS.ZS: 9576 of 9576
IE.PPI.ICTI.CD: 9576 of 9576
FX.OWN.TOTL.ZS: 9576 of 9576
FX.OWN.TOTL.FE.ZS: 9576 of 9576
SL.GDP.PCAP.EM.KD: 9576 of 9576
SL.TLF.CACT.FE.ZS: 9576 of 9576
SL.TLF.CACT.FM.ZS: 9576 of 9576
NY.GDP.PCAP.PP.KD: 9576 of 9576
SE.TER.ENRR: 9576 of 9576
NE.TRD.GNFS.ZS: 9576 of 9576
SP.URB.TOTL.IN.ZS: 9576 of 9576
EG.ELC.ACCS.ZS: 9576 of 9576
GOV_WGI_GE_EST: 9576 of 9576
GOV_WGI_RQ_EST: 9576 of 9576
Total observations: 92105


In [21]:
df_long = pd.read_csv("panel_long.csv")
print(df_long.shape)
print(df_long["indicator"].value_counts())

(92105, 4)
indicator
SP.URB.TOTL.IN.ZS    7812
IT.CEL.SETS.P2       7029
NY.GDP.PCAP.PP.KD    6977
EG.ELC.ACCS.ZS       6747
SL.TLF.CACT.FE.ZS    6718
SL.TLF.CACT.FM.ZS    6718
IT.NET.USER.ZS       6224
NE.TRD.GNFS.ZS       6134
SL.GDP.PCAP.EM.KD    6119
GOV_WGI_RQ_EST       5374
GOV_WGI_GE_EST       5373
BX.GSR.CCIS.ZS       5073
IT.NET.BBND.P2       4330
SE.TER.ENRR          3863
IT.NET.SECR.P6       3193
IT.NET.USER.MA.ZS    1363
IT.NET.USER.FE.ZS    1362
FX.OWN.TOTL.ZS        705
FX.OWN.TOTL.FE.ZS     679
IE.PPI.ICTI.CD        312
Name: count, dtype: int64


In [22]:
df_long["year"] = df_long["year"].astype(int)

for code in ["IT.NET.USER.FE.ZS", "IT.NET.USER.MA.ZS", "FX.OWN.TOTL.FE.ZS", "IT.NET.USER.ZS", "SL.TLF.CACT.FE.ZS", "SL.GDP.PCAP.EM.KD"]:
    sub = df_long[df_long["indicator"] == code]
    print("---", code, "| countries:", sub["country"].nunique(), "| years:", sub["year"].min(), "-", sub["year"].max())
    print(sub.groupby("year")["country"].count().tail(15))

--- IT.NET.USER.FE.ZS | countries: 137 | years: 2000 - 2025
year
2011    51
2012    59
2013    67
2014    73
2015    78
2016    76
2017    92
2018    82
2019    92
2020    74
2021    77
2022    70
2023    67
2024    65
2025     7
Name: country, dtype: int64
--- IT.NET.USER.MA.ZS | countries: 137 | years: 2000 - 2025
year
2011    51
2012    59
2013    68
2014    73
2015    78
2016    76
2017    92
2018    82
2019    92
2020    74
2021    77
2022    70
2023    67
2024    65
2025     7
Name: country, dtype: int64
--- FX.OWN.TOTL.FE.ZS | countries: 159 | years: 2011 - 2024
year
2011    126
2014    135
2017    143
2021    120
2022     16
2024    139
Name: country, dtype: int64
--- IT.NET.USER.ZS | countries: 213 | years: 1990 - 2025
year
2011    204
2012    203
2013    201
2014    201
2015    201
2016    203
2017    200
2018    182
2019    189
2020    184
2021    180
2022    182
2023    181
2024    182
2025     10
Name: country, dtype: int64
--- SL.TLF.CACT.FE.ZS | countries: 187 | years: 1

In [23]:
df_long["year"] = df_long["year"].astype(int)

key = [
    "IT.NET.USER.ZS", "IT.NET.USER.FE.ZS", "IT.NET.USER.MA.ZS",
    "SL.GDP.PCAP.EM.KD", "SL.TLF.CACT.FE.ZS",
    "NY.GDP.PCAP.PP.KD", "SE.TER.ENRR", "NE.TRD.GNFS.ZS",
    "SP.URB.TOTL.IN.ZS", "EG.ELC.ACCS.ZS",
    "GOV_WGI_GE_EST", "GOV_WGI_RQ_EST",
]

wide = df_long[df_long["indicator"].isin(key)].pivot_table(
    index=["country", "year"], columns="indicator", values="value"
).reset_index()

for lo, hi in [(2011, 2023), (2011, 2024), (2015, 2024), (2017, 2024)]:
    sub = wide[(wide["year"] >= lo) & (wide["year"] <= hi)].dropna()
    print(f"{lo}-{hi}: rows={len(sub)}, countries={sub['country'].nunique()}, mean years/country={len(sub)/max(sub['country'].nunique(),1):.1f}")
    

2011-2023: rows=837, countries=106, mean years/country=7.9
2011-2024: rows=876, countries=107, mean years/country=8.2
2015-2024: rows=671, countries=107, mean years/country=6.3
2017-2024: rows=532, countries=105, mean years/country=5.1


In [24]:
import pandas as pd
df_long["year"] = df_long["year"].astype(int)

key = [
    "IT.NET.USER.ZS", "IT.NET.USER.FE.ZS", "IT.NET.USER.MA.ZS",
    "SL.GDP.PCAP.EM.KD", "SL.TLF.CACT.FE.ZS",
    "NY.GDP.PCAP.PP.KD", "SE.TER.ENRR", "NE.TRD.GNFS.ZS",
    "SP.URB.TOTL.IN.ZS", "EG.ELC.ACCS.ZS",
    "GOV_WGI_GE_EST", "GOV_WGI_RQ_EST",
]

wide = df_long[df_long["indicator"].isin(key)].pivot_table(
    index=["country", "year"], columns="indicator", values="value"
).reset_index()

window = wide[(wide["year"] >= 2011) & (wide["year"] <= 2024)].dropna()
window.to_csv("panel_window_2011_2024.csv", index=False)
print("saved:", window.shape)

coverage = df_long.groupby("indicator").agg(
    n_obs=("value", "count"),
    n_countries=("country", "nunique"),
    year_min=("year", "min"),
    year_max=("year", "max"),
).sort_values("n_obs", ascending=False)
coverage.to_csv("coverage_by_indicator.csv")
print(coverage)

saved: (876, 14)
                   n_obs  n_countries  year_min  year_max
indicator                                                
SP.URB.TOTL.IN.ZS   7812          217      1990      2025
IT.CEL.SETS.P2      7029          214      1990      2024
NY.GDP.PCAP.PP.KD   6977          199      1990      2025
EG.ELC.ACCS.ZS      6747          216      1990      2024
SL.TLF.CACT.FM.ZS   6718          187      1990      2025
SL.TLF.CACT.FE.ZS   6718          187      1990      2025
IT.NET.USER.ZS      6224          213      1990      2025
NE.TRD.GNFS.ZS      6134          193      1990      2025
SL.GDP.PCAP.EM.KD   6119          177      1991      2025
GOV_WGI_RQ_EST      5374          205      1996      2025
GOV_WGI_GE_EST      5373          205      1996      2025
BX.GSR.CCIS.ZS      5073          190      1990      2025
IT.NET.BBND.P2      4330          208      1998      2024
SE.TER.ENRR         3863          200      1990      2025
IT.NET.SECR.P6      3193          215      2010      20

In [25]:
# PARKED: Task 0 redo from scratch
# Goal: rebuild the whole Task 0 pipeline without hints.
# Files to reproduce:
#   countries_real.json
#   indicators_catalog.csv
#   shortlist_codes.txt
#   panel_long.csv
#   panel_window_2011_2024.csv
#   coverage_by_indicator.csv
# Key gotchas I hit:
#   - API default per_page = 50, must set higher
#   - API default returns latest value per country, must add &date=1990:2025
#   - Indicator catalog is paginated, loop pages
#   - country/region field: exclude region == "Aggregates"
#   - countryiso3code not country.id
# Attempt date: Oct 11

In [26]:
rows = []
for kw in keywords:
    hits = wdi[wdi["indicator_name"].str.contains(re.escape(kw), case=False, na=False)]
    for _, r in hits.iterrows():
        rows.append({
            "keyword": kw,
            "code": r["indicator_id"],
            "name": r["indicator_name"],
            "decision": "keep" if r["indicator_id"] in shortlist else "drop",
        })

decisions = pd.DataFrame(rows)
print(decisions.shape)
print(decisions.head(15).to_string())

(56, 4)
                                         keyword               code                                                                                                                          name decision
0                 Individuals using the Internet  IT.NET.USER.FE.ZS                                                               Individuals using the Internet, female (% of female population)     keep
1                 Individuals using the Internet  IT.NET.USER.MA.ZS                                                                   Individuals using the Internet, male (% of male population)     keep
2                 Individuals using the Internet     IT.NET.USER.ZS                                                                              Individuals using the Internet (% of population)     keep
3                  Fixed broadband subscriptions        IT.NET.BBND                                                                                                 Fixed broadband 

In [27]:
print(decisions["decision"].value_counts())

decision
drop    36
keep    20
Name: count, dtype: int64


In [28]:
reasons = {
    # --- Keeps: adoption ---
    "IT.NET.USER.ZS":     "Total internet penetration. Primary digital adoption measure.",
    "IT.NET.USER.FE.ZS":  "Female internet penetration. Required to construct the gender gap.",
    "IT.NET.USER.MA.ZS":  "Male internet penetration. Required to construct the gender gap.",
    "IT.NET.BBND.P2":     "Fixed broadband per 100. Rate form for cross-country comparability.",
    "IT.CEL.SETS.P2":     "Mobile subscriptions per 100. Rate form for cross-country comparability.",
    "IT.NET.SECR.P6":     "Secure servers per 1m. Rate form for cross-country comparability.",
    "BX.GSR.CCIS.ZS":     "ICT service exports as share of service exports. Scale-free, comparable.",
    "IE.PPI.ICTI.CD":     "ICT investment with private participation. Only available variant.",

    # --- Keeps: inclusion ---
    "FX.OWN.TOTL.ZS":     "Account ownership, population 15+. Total aggregate.",
    "FX.OWN.TOTL.FE.ZS":  "Account ownership, female 15+. Required for inclusion analysis.",

    # --- Keeps: outcomes ---
    "SL.GDP.PCAP.EM.KD":  "GDP per person employed, constant 2021 PPP. Primary productivity outcome.",
    "SL.TLF.CACT.FE.ZS":  "Female LFP, ages 15+, ILO modelled. Primary participation outcome.",
    "SL.TLF.CACT.FM.ZS":  "Female-to-male LFP ratio, ILO modelled. Secondary participation outcome.",

    # --- Keeps: controls ---
    "NY.GDP.PCAP.PP.KD":  "GDP per capita, constant PPP. Income control.",
    "SE.TER.ENRR":        "Tertiary gross enrolment. Human capital control.",
    "NE.TRD.GNFS.ZS":     "Trade as share of GDP. Openness control.",
    "SP.URB.TOTL.IN.ZS":  "Urban population share. Structural control.",
    "EG.ELC.ACCS.ZS":     "Electricity access, total population. Infrastructure control.",
    "GOV_WGI_GE_EST":     "Government effectiveness, estimate scale. Institutional control.",
    "GOV_WGI_RQ_EST":     "Regulatory quality, estimate scale. Institutional control.",

    # --- Drops: level counts, not normalised ---
    "IT.NET.BBND":        "Level count. Retained rate form for cross-country comparability.",
    "IT.CEL.SETS":        "Level count. Retained rate form for cross-country comparability.",
    "IT.NET.SECR":        "Level count. Retained rate form for cross-country comparability.",
    "BX.GSR.CCIS.CD":     "Nominal USD value. Retained share-of-exports form for comparability.",

    # --- Drops: sub-population splits, out of scope ---
    "FX.OWN.TOTL.40.ZS":  "Income-quintile split. Scope restricted to total and female aggregates.",
    "FX.OWN.TOTL.60.ZS":  "Income-quintile split. Scope restricted to total and female aggregates.",
    "FX.OWN.TOTL.MA.ZS":  "Male-only split. Inclusion analysis uses total and female aggregates.",
    "FX.OWN.TOTL.OL.ZS":  "Age split (25+). Out of scope.",
    "FX.OWN.TOTL.PL.ZS":  "Education split (primary or less). Out of scope.",
    "FX.OWN.TOTL.SO.ZS":  "Education split (secondary or more). Out of scope.",
    "FX.OWN.TOTL.YG.ZS":  "Age split (15-24). Out of scope.",

    # --- Drops: alternative definitions of shortlisted constructs ---
    "SL.TLF.ACTI.FE.ZS":  "Age band 15-64. Retained 15+ headline series for consistency.",
    "SL.TLF.CACT.FE.NE.ZS":"National estimate. Retained ILO modelled series for cross-country comparability.",
    "SL.TLF.CACT.FM.NE.ZS":"National estimate. Retained ILO modelled series for cross-country comparability.",
    "NY.GDP.PCAP.PP.CD":  "Current international prices. Retained constant-price series for real comparisons.",
    "SE.ENR.TERT.FM.ZS":  "Gender parity index. Retained gross enrolment rate.",
    "SE.TER.ENRR.FE":     "Female-only enrolment. Retained total enrolment for control specification.",
    "SE.TER.ENRR.MA":     "Male-only enrolment. Retained total enrolment for control specification.",
    "EG.ELC.ACCS.RU.ZS":  "Rural-only access. Retained total-population access.",
    "EG.ELC.ACCS.UR.ZS":  "Urban-only access. Retained total-population access.",
    "GOV_WGI_GE_SC":      "Rescaled score (0-100). Retained estimate scale for continuity with WGI.",
    "GOV_WGI_RQ_SC":      "Rescaled score (0-100). Retained estimate scale for continuity with WGI.",

    # --- Drops: adjacent but distinct constructs ---
    "TG.VAL.TOTL.GD.ZS":  "Merchandise trade only. Shortlisted measure is total trade share.",

    # --- Drops: keyword collisions on 'urban population' ---
    "EG.CFT.ACCS.UR.ZS":  "Cooking-fuel access, urban. Not an urbanisation measure.",
    "EN.POP.EL5M.UR.ZS":  "Low-elevation population. Not an urbanisation measure.",
    "EN.POP.SLUM.UR.ZS":  "Slum population. Not an urbanisation measure.",
    "EN.URB.LCTY.UR.ZS":  "Largest-city concentration. Not the urban share of total population.",
    "SH.H2O.BASW.UR.ZS":  "Basic drinking water access. Not an urbanisation measure.",
    "SH.H2O.SMDW.UR.ZS":  "Safely managed drinking water. Not an urbanisation measure.",
    "SH.STA.BASS.UR.ZS":  "Basic sanitation access. Not an urbanisation measure.",
    "SH.STA.HYGN.UR.ZS":  "Handwashing facility access. Not an urbanisation measure.",
    "SH.STA.ODFC.UR.ZS":  "Open defecation. Not an urbanisation measure.",
    "SH.STA.SMSS.UR.ZS":  "Safely managed sanitation. Not an urbanisation measure.",
    "SP.URB.GROW":        "Urban population growth rate. Retained urban share of total.",
    "SP.URB.TOTL":        "Urban population level. Retained urban share of total.",
}

decisions["reason"] = decisions["code"].map(reasons)
print(decisions["reason"].isna().sum(), "rows without a reason")
print(decisions.shape)
print(decisions["decision"].value_counts())
decisions.to_csv("shortlist_decisions.csv", index=False)
print(decisions.head(20).to_string())

0 rows without a reason
(56, 5)
decision
drop    36
keep    20
Name: count, dtype: int64
                                         keyword               code                                                                                                                                          name decision                                                                    reason
0                 Individuals using the Internet  IT.NET.USER.FE.ZS                                                                               Individuals using the Internet, female (% of female population)     keep        Female internet penetration. Required to construct the gender gap.
1                 Individuals using the Internet  IT.NET.USER.MA.ZS                                                                                   Individuals using the Internet, male (% of male population)     keep          Male internet penetration. Required to construct the gender gap.
2                 Individuals us

In [29]:
import pandas as pd
w = pd.read_csv("panel_window_2011_2024.csv")
print(w.shape)
print(w.columns.tolist())
print(w.head())
print(w.groupby("country").size().describe())

(876, 14)
['country', 'year', 'EG.ELC.ACCS.ZS', 'GOV_WGI_GE_EST', 'GOV_WGI_RQ_EST', 'IT.NET.USER.FE.ZS', 'IT.NET.USER.MA.ZS', 'IT.NET.USER.ZS', 'NE.TRD.GNFS.ZS', 'NY.GDP.PCAP.PP.KD', 'SE.TER.ENRR', 'SL.GDP.PCAP.EM.KD', 'SL.TLF.CACT.FE.ZS', 'SP.URB.TOTL.IN.ZS']
  country  year  EG.ELC.ACCS.ZS  GOV_WGI_GE_EST  GOV_WGI_RQ_EST  \
0     ALB  2019           100.0        0.115817        0.163741   
1     ALB  2020           100.0       -0.009337        0.059186   
2     ALB  2021           100.0        0.108063        0.056966   
3     ALB  2022           100.0        0.192283        0.059804   
4     ALB  2023           100.0        0.264196        0.104090   

   IT.NET.USER.FE.ZS  IT.NET.USER.MA.ZS  IT.NET.USER.ZS  NE.TRD.GNFS.ZS  \
0            65.9616            71.1594       68.550391       75.382129   
1            71.2457            73.2383       72.237677       59.520699   
2            78.3496            80.3109       79.323718       75.590404   
3            81.5432            83.7

In [30]:
years_per_country = w.groupby("country").size()

print("Countries with 1 year:", (years_per_country == 1).sum())
print("Countries with 2-3 years:", ((years_per_country >= 2) & (years_per_country <= 3)).sum())
print("Countries with 4-7 years:", ((years_per_country >= 4) & (years_per_country <= 7)).sum())
print("Countries with 8+ years:", (years_per_country >= 8).sum())
print()
print("Total countries:", len(years_per_country))

Countries with 1 year: 13
Countries with 2-3 years: 14
Countries with 4-7 years: 15
Countries with 8+ years: 65

Total countries: 107


In [31]:
years_per_country = w.groupby("country").size()
keep_countries = years_per_country[years_per_country >= 3].index

w_filtered = w[w["country"].isin(keep_countries)].copy()
w_filtered = w_filtered.sort_values(["country", "year"])

print("Countries kept:", w_filtered["country"].nunique())
print("Rows kept:", len(w_filtered))
print("Years per country after filter:")
print(w_filtered.groupby("country").size().describe())

Countries kept: 86
Rows kept: 847
Years per country after filter:
count    86.000000
mean      9.848837
std       3.618636
min       3.000000
25%       8.000000
50%      10.000000
75%      13.000000
max      14.000000
dtype: float64


In [32]:
# gender gap
w_filtered["gender_gap"] = (
    w_filtered["IT.NET.USER.MA.ZS"] - w_filtered["IT.NET.USER.FE.ZS"]
)

# lagged digital adoption within country
w_filtered = w_filtered.sort_values(["country", "year"]).copy()
w_filtered["IT_lag1"] = (
    w_filtered.groupby("country")["IT.NET.USER.ZS"].shift(1)
)

# report what we have
print("Rows before lag drop:", len(w_filtered))
print("Rows with non-null lag:", w_filtered["IT_lag1"].notna().sum())
print()
print("Gender gap summary:")
print(w_filtered["gender_gap"].describe())
print()
print(w_filtered[["country", "year", "IT.NET.USER.ZS", "IT_lag1",
                  "IT.NET.USER.MA.ZS", "IT.NET.USER.FE.ZS", "gender_gap"]].head(10))

Rows before lag drop: 847
Rows with non-null lag: 761

Gender gap summary:
count    847.000000
mean       2.786897
std        3.936985
min      -20.010900
25%        0.370200
50%        2.058900
75%        4.351700
max       33.803800
Name: gender_gap, dtype: float64

  country  year  IT.NET.USER.ZS     IT_lag1  IT.NET.USER.MA.ZS  \
0     ALB  2019       68.550391         NaN            71.1594   
1     ALB  2020       72.237677   68.550391            73.2383   
2     ALB  2021       79.323718   72.237677            80.3109   
3     ALB  2022       82.613675   79.323718            83.7135   
4     ALB  2023       83.135592   82.613675            84.5900   
5     ARE  2018       98.450002         NaN            98.3353   
6     ARE  2019       99.149998   98.450002            99.0300   
7     ARE  2020      100.000000   99.149998           100.0000   
8     ARE  2021      100.000000  100.000000           100.0000   
9     ARE  2022      100.000000  100.000000           100.0000   

   I

In [33]:
w_filtered.to_csv("panel_task2.csv", index=False)

# inspect tails of gender gap
print("Most negative gender gaps (women ahead):")
print(w_filtered.nsmallest(5, "gender_gap")[
    ["country", "year", "IT.NET.USER.MA.ZS", "IT.NET.USER.FE.ZS", "gender_gap"]
])

print()
print("Most positive gender gaps (men ahead):")
print(w_filtered.nlargest(5, "gender_gap")[
    ["country", "year", "IT.NET.USER.MA.ZS", "IT.NET.USER.FE.ZS", "gender_gap"]
])

# how many rows are at the 100% ceiling
print()
print("Rows where total internet use is 100:", (w_filtered["IT.NET.USER.ZS"] == 100).sum())

Most negative gender gaps (women ahead):
    country  year  IT.NET.USER.MA.ZS  IT.NET.USER.FE.ZS  gender_gap
85      BHR  2011            70.0000            90.0109    -20.0109
606     MNG  2018            42.7963            51.2040     -8.4077
607     MNG  2019            46.8610            55.0473     -8.1863
128     BRA  2020            77.3330            84.9105     -7.5775
87      BHR  2014            87.9999            95.0001     -7.0002

Most positive gender gaps (men ahead):
    country  year  IT.NET.USER.MA.ZS  IT.NET.USER.FE.ZS  gender_gap
156     CIV  2017            60.2019            26.3981     33.8038
832     TUR  2012            55.8033            34.6829     21.1204
833     TUR  2013            56.6006            36.0678     20.5328
549     MAR  2012            65.3959            45.7717     19.6242
835     TUR  2015            63.5388            44.0396     19.4992

Rows where total internet use is 100: 11


In [34]:
pip install linearmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 1.6 MB/s  0:00:00m 5.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [linearmodels]0m 3/4 [linearmodels]
Note: you may need to restart the kernel to use updated packages.


In [3]:
from linearmodels.panel import PanelOLS

df = pd.read_csv("panel_task2.csv")
df = df.dropna(subset=["IT_lag1"])
df = df.set_index(["country", "year"])

outcome = "SL.GDP.PCAP.EM.KD"
#GDP per person employed, constant 2021 PPP dollars.
regressors = [
    "IT_lag1",
    "NY.GDP.PCAP.PP.KD",
    "SE.TER.ENRR",
    "NE.TRD.GNFS.ZS",
    "SP.URB.TOTL.IN.ZS",
    "EG.ELC.ACCS.ZS",
    "GOV_WGI_GE_EST",
    "GOV_WGI_RQ_EST",
]
#IT_lag1 — internet penetration, lagged one year, as a percentage of the population. Calculated above
#NY.GDP.PCAP.PP.KD — GDP per capita, constant 2021 PPP dollars.
#SE.TER.ENRR — tertiary school enrolment, gross percentage
#NE.TRD.GNFS.ZS — trade as a percentage of GDP.
#SP.URB.TOTL.IN.ZS — urban population as a percentage of total population.
#EG.ELC.ACCS.ZS — access to electricity, percentage of population.
#GOV_WGI_GE_EST — government effectiveness, WGI estimate scale (roughly -2.5 to +2.5)
#GOV_WGI_RQ_EST — regulatory quality, WGI estimate scale (roughly -2.5 to +2.5)
y = df[outcome]
X = df[regressors]

mod = PanelOLS(y, X, entity_effects=True, time_effects=True)
res = mod.fit(cov_type="clustered", cluster_entity=True)

print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:      SL.GDP.PCAP.EM.KD   R-squared:                        0.8958
Estimator:                   PanelOLS   R-squared (Between):              0.9210
No. Observations:                 761   R-squared (Within):               0.8849
Date:                Tue, Oct 13 2026   R-squared (Overall):              0.9354
Time:                        11:46:22   Log-likelihood                   -6956.2
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      703.84
Entities:                          86   P-value                           0.0000
Avg Obs:                       8.8488   Distribution:                   F(8,655)
Min Obs:                       2.0000                                           
Max Obs:                       13.000   F-statistic (robust):             177.41
                            

In [4]:
with open("results_productivity.txt", "w") as f:
    f.write(str(res.summary))

In [5]:
outcome = "SL.TLF.CACT.FE.ZS"
# outcome is Labor force participation rate, female, ages 15+, modeled ILO estimate.
y = df[outcome]
X = df[regressors]

mod_lfp = PanelOLS(y, X, entity_effects=True, time_effects=True)
res_lfp = mod_lfp.fit(cov_type="clustered", cluster_entity=True)

print(res_lfp.summary)

with open("results_flfp.txt", "w") as f:
    f.write(str(res_lfp.summary))

                          PanelOLS Estimation Summary                           
Dep. Variable:      SL.TLF.CACT.FE.ZS   R-squared:                        0.0534
Estimator:                   PanelOLS   R-squared (Between):             -1.4484
No. Observations:                 761   R-squared (Within):               0.0230
Date:                Tue, Oct 13 2026   R-squared (Overall):             -1.4589
Time:                        11:55:13   Log-likelihood                   -1346.7
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      4.6202
Entities:                          86   P-value                           0.0000
Avg Obs:                       8.8488   Distribution:                   F(8,655)
Min Obs:                       2.0000                                           
Max Obs:                       13.000   F-statistic (robust):             1.0582
                            

In [6]:
with open("results_flfp.txt", "w") as f:
    f.write(str(res_lfp.summary))

In [8]:
df["IT_lag1_x_gap"] = df["IT_lag1"] * df["gender_gap"]

interaction_regs = [
    "IT_lag1",
    "gender_gap",
    "IT_lag1_x_gap",
    "NY.GDP.PCAP.PP.KD",
    "SE.TER.ENRR",
    "NE.TRD.GNFS.ZS",
    "SP.URB.TOTL.IN.ZS",
    "EG.ELC.ACCS.ZS",
    "GOV_WGI_GE_EST",
    "GOV_WGI_RQ_EST",
]

y = df["SL.TLF.CACT.FE.ZS"]
X = df[interaction_regs]

mod_int = PanelOLS(y, X, entity_effects=True, time_effects=True)
res_int = mod_int.fit(cov_type="clustered", cluster_entity=True)

print(res_int.summary)

with open("results_interaction.txt", "w") as f:
    f.write(str(res_int.summary))

                          PanelOLS Estimation Summary                           
Dep. Variable:      SL.TLF.CACT.FE.ZS   R-squared:                        0.0638
Estimator:                   PanelOLS   R-squared (Between):             -1.3250
No. Observations:                 761   R-squared (Within):               0.0740
Date:                Tue, Oct 13 2026   R-squared (Overall):             -1.3335
Time:                        11:58:26   Log-likelihood                   -1342.5
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      4.4464
Entities:                          86   P-value                           0.0000
Avg Obs:                       8.8488   Distribution:                  F(10,653)
Min Obs:                       2.0000                                           
Max Obs:                       13.000   F-statistic (robust):             0.9527
                            

In [11]:



df = pd.read_csv("panel_task2.csv")
df = df.dropna(subset=["IT_lag1"])
df = df.set_index(["country", "year"])

regressors = [
    "IT_lag1",
    "NY.GDP.PCAP.PP.KD",
    "SE.TER.ENRR",
    "NE.TRD.GNFS.ZS",
    "SP.URB.TOTL.IN.ZS",
    "EG.ELC.ACCS.ZS",
    "GOV_WGI_GE_EST",
    "GOV_WGI_RQ_EST",
]

results = []

for outcome in ["SL.GDP.PCAP.EM.KD", "SL.TLF.CACT.FE.ZS"]:
    y = df[outcome]
    X = df[regressors]

    pooled = PooledOLS(y, X).fit(cov_type="clustered", cluster_entity=True)
    results.append({
        "outcome": outcome, "spec": "pooled",
        "coef": pooled.params["IT_lag1"],
        "se": pooled.std_errors["IT_lag1"],
        "pval": pooled.pvalues["IT_lag1"],
    })

    fe = PanelOLS(y, X, entity_effects=True, time_effects=True).fit(
        cov_type="clustered", cluster_entity=True
    )
    results.append({
        "outcome": outcome, "spec": "fe",
        "coef": fe.params["IT_lag1"],
        "se": fe.std_errors["IT_lag1"],
        "pval": fe.pvalues["IT_lag1"],
    })

out = pd.DataFrame(results)
out.to_csv("python_coef_table.csv", index=False)
print(out.to_string(index=False))

          outcome   spec       coef        se     pval
SL.GDP.PCAP.EM.KD pooled -57.617001 69.878894 0.409902
SL.GDP.PCAP.EM.KD     fe  55.638554 33.291138 0.095146
SL.TLF.CACT.FE.ZS pooled   0.058937  0.063654 0.354790
SL.TLF.CACT.FE.ZS     fe  -0.017538  0.021462 0.414137


In [12]:
import os
print(os.path.exists("python_coef_table.csv"))

True


In [16]:


python = pd.read_csv("python_coef_table.csv")
print("Python:")
print(python.to_string(index=False))

print()
for f in ["r_results_productivity.txt", "r_results_flfp.txt", "r_results_interaction.txt"]:
    print(f"R: {f}")
    with open(f) as fh:
        print(fh.read())
    print("---")   

Python:
          outcome   spec       coef        se     pval
SL.GDP.PCAP.EM.KD pooled -57.617001 69.878894 0.409902
SL.GDP.PCAP.EM.KD     fe  55.638554 33.291138 0.095146
SL.TLF.CACT.FE.ZS pooled   0.058937  0.063654 0.354790
SL.TLF.CACT.FE.ZS     fe  -0.017538  0.021462 0.414137

R: r_results_productivity.txt
                              m_prod
Dependent Var.:    SL.GDP.PCAP.EM.KD
                                    
IT_lag1               55.64. (31.48)
NY.GDP.PCAP.PP.KD  1.579*** (0.0522)
SE.TER.ENRR            7.163 (48.71)
NE.TRD.GNFS.ZS       -45.20. (24.94)
SP.URB.TOTL.IN.ZS      159.2 (162.4)
EG.ELC.ACCS.ZS        313.5. (167.4)
GOV_WGI_GE_EST     1,669.9 (2,851.5)
GOV_WGI_RQ_EST    -2,325.9 (1,933.5)
Fixed-Effects:    ------------------
country                          Yes
year                             Yes
_________________ __________________
S.E.: Clustered          by: country
Observations                     761
R2                           0.99781
Within R2           

In [17]:
stata = pd.read_csv("stata_coef_table.txt", sep="\t", skiprows=1, on_bad_lines="skip", engine="python")
print(stata.to_string())

                               Unnamed: 0         m_prod          m_lfp          m_int
0                               VARIABLES  slgdppcapemkd  sltlfcactfezs  sltlfcactfezs
1                                     NaN            NaN            NaN            NaN
2                                 it_lag1         55.64*        -0.0175        -0.0119
3                                     NaN        (31.48)       (0.0203)       (0.0262)
4                              gender_gap            NaN            NaN         0.0633
5                                     NaN            NaN            NaN        (0.218)
6                           it_lag1_x_gap            NaN            NaN       -0.00208
7                                     NaN            NaN            NaN      (0.00335)
8                           nygdppcapppkd       1.579***       4.03e-05       3.91e-05
9                                     NaN       (0.0522)     (3.02e-05)     (2.94e-05)
10                              seterenrr  

In [19]:
import re
stata = pd.read_csv("stata_coef_table.txt", sep="\t", header=0, engine="python", skipinitialspace=True)
stata.columns = ["var", "m_prod", "m_lfp", "m_int"]

def grab(var_name, col):
    row = stata[stata["var"] == var_name]
    if row.empty:
        return None, None
    coef_str = str(row[col].values[0])
    idx = row.index[0]
    se_str = str(stata.iloc[idx + 1][col]) if idx + 1 < len(stata) else ""
    coef = float(re.sub(r"[^\d\.\-]", "", coef_str.split("(")[0].strip()) or "nan")
    se_match = re.search(r"\(([^)]+)\)", se_str)
    se = float(re.sub(r"[^\d\.\-]", "", se_match.group(1))) if se_match else None
    return coef, se

rows = [
    {"outcome": "SL.GDP.PCAP.EM.KD", "spec": "fe",             "term": "it_lag1",       "stata_coef": grab("it_lag1", "m_prod")[0],       "stata_se": grab("it_lag1", "m_prod")[1]},
    {"outcome": "SL.TLF.CACT.FE.ZS", "spec": "fe",             "term": "it_lag1",       "stata_coef": grab("it_lag1", "m_lfp")[0],        "stata_se": grab("it_lag1", "m_lfp")[1]},
    {"outcome": "SL.TLF.CACT.FE.ZS", "spec": "fe_interaction", "term": "it_lag1_x_gap", "stata_coef": grab("it_lag1_x_gap", "m_int")[0],  "stata_se": grab("it_lag1_x_gap", "m_int")[1]},
]

s = pd.DataFrame(rows)
print(s.to_string(index=False))
s.to_csv("stata_coef_table_clean.csv", index=False)

          outcome           spec          term  stata_coef  stata_se
SL.GDP.PCAP.EM.KD             fe       it_lag1    55.64000  31.48000
SL.TLF.CACT.FE.ZS             fe       it_lag1    -0.01750   0.02030
SL.TLF.CACT.FE.ZS fe_interaction it_lag1_x_gap    -0.00208   0.00335


In [21]:


def parse_r_line(line):
    # line looks like "IT_lag1               55.64. (31.48)"
    parts = line.strip().split()
    # parts[0] = "IT_lag1", parts[1] = "55.64.", parts[2] = "(31.48)"
    coef_raw = parts[1]
    # strip anything that is not a digit, minus, or dot
    coef_clean = re.sub(r"[^\d\.\-]", "", coef_raw)
    # remove trailing dots
    coef_clean = coef_clean.rstrip(".")
    coef = float(coef_clean)
    se_match = re.search(r"\(([^)]+)\)", line)
    se = float(se_match.group(1)) if se_match else None
    return coef, se

def read_r_coef(path, term, outcome, spec):
    with open(path) as f:
        for line in f:
            stripped = line.strip()
            if stripped.startswith(term):
                coef, se = parse_r_line(stripped)
                return {"outcome": outcome, "spec": spec, "term": term,
                        "r_coef": coef, "r_se": se}
    return None

r_rows = [
    read_r_coef("r_results_productivity.txt", "IT_lag1", "SL.GDP.PCAP.EM.KD", "fe"),
    read_r_coef("r_results_flfp.txt", "IT_lag1", "SL.TLF.CACT.FE.ZS", "fe"),
    read_r_coef("r_results_interaction.txt", "IT_lag1_x_gap", "SL.TLF.CACT.FE.ZS", "fe_interaction"),
]
r = pd.DataFrame(r_rows)
print(r.to_string(index=False))

          outcome           spec          term  r_coef    r_se
SL.GDP.PCAP.EM.KD             fe       IT_lag1 55.6400 31.4800
SL.TLF.CACT.FE.ZS             fe       IT_lag1 -0.0175  0.0203
SL.TLF.CACT.FE.ZS fe_interaction IT_lag1_x_gap -0.0021  0.0033


In [23]:
import pandas as pd

# --- Python ---
python = pd.read_csv("python_coef_table.csv")

py_fe = python[python["spec"] == "fe"][["outcome", "coef", "se"]].copy()
py_fe["spec"] = "fe"
py_fe["term"] = "IT_lag1"
py_fe = py_fe.rename(columns={"coef": "py_coef", "se": "py_se"})

py_int = pd.DataFrame([{
    "outcome": "SL.TLF.CACT.FE.ZS", "spec": "fe_interaction",
    "term": "IT_lag1_x_gap", "py_coef": -0.0021, "py_se": 0.0035,
}])
py = pd.concat([py_fe, py_int], ignore_index=True)

# --- R (already parsed in previous cell as r) ---
# --- Stata ---
s = pd.read_csv("stata_coef_table_clean.csv")
s = s.rename(columns={"stata_coef": "stata_coef", "stata_se": "stata_se"})

# --- Merge ---
out = py.merge(r[["outcome", "spec", "r_coef", "r_se"]], on=["outcome", "spec"], how="left")
out = out.merge(s[["outcome", "spec", "stata_coef", "stata_se"]], on=["outcome", "spec"], how="left")

out = out[["outcome", "spec", "term",
           "py_coef", "r_coef", "stata_coef",
           "py_se", "r_se", "stata_se"]]

out.to_csv("task3_reconciliation.csv", index=False)
print(out.to_string(index=False))

          outcome           spec          term   py_coef  r_coef  stata_coef     py_se    r_se  stata_se
SL.GDP.PCAP.EM.KD             fe       IT_lag1 55.638554 55.6400    55.64000 33.291138 31.4800  31.48000
SL.TLF.CACT.FE.ZS             fe       IT_lag1 -0.017538 -0.0175    -0.01750  0.021462  0.0203   0.02030
SL.TLF.CACT.FE.ZS fe_interaction IT_lag1_x_gap -0.002100 -0.0021    -0.00208  0.003500  0.0033   0.00335
